In [6]:
import math
import re
import csv
import os

def leer_y_ordenar_ruido_ngspice(ruta_archivo):
    if not os.path.exists(ruta_archivo):
        raise FileNotFoundError(f"No se encontró el archivo en: {ruta_archivo}")

    with open(ruta_archivo, 'rb') as f:
        contenido = f.read()

    # Separar cabecera de datos (Values: o Binary:)
    match_datos = re.search(rb'(Values:|Binary:)\s*\r?\n', contenido)
    if not match_datos:
        raise ValueError("No se encontró 'Values:' o 'Binary:' en el archivo.")

    cabecera = contenido[:match_datos.start()].decode('utf-8', errors='ignore')
    tipo_datos = match_datos.group(1).decode('utf-8')
    cuerpo_datos = contenido[match_datos.end():]

    # Extraer número de variables y puntos
    coincidencia_vars = re.search(r'No\. Variables:\s*(\d+)', cabecera)
    if coincidencia_vars is None:
        raise ValueError("La cabecera no contiene 'No. Variables:'.")
    num_vars = int(coincidencia_vars.group(1))
    coincidencia_puntos = re.search(r'No\. Points:\s*(\d+)', cabecera)
    num_puntos = int(coincidencia_puntos.group(1)) if coincidencia_puntos else None
    plotname_match = re.search(r'Plotname:\s*(.+)', cabecera)
    plotname = plotname_match.group(1).strip() if plotname_match else ''

    # Extraer nombres de variables (identificadores)
    seccion_vars = cabecera.split('Variables:', 1)[-1]
    variables = []
    for linea in seccion_vars.strip().splitlines():
        partes = linea.strip().split()
        if len(partes) >= 2 and partes[0].isdigit():
            variables.append(partes[1])

    if tipo_datos != 'Values:':
        raise NotImplementedError(
            "Este lector admite el formato ASCII ('Values:'). "
            "Exporta el raw con 'set filetype=ascii' en ngspice."
        )

    # En cada punto ngspice escribe: índice, variable 0, ..., variable N-1.
    tokens = cuerpo_datos.decode('utf-8', errors='strict').split()
    tam_punto = num_vars + 1
    if len(tokens) % tam_punto != 0:
        raise ValueError(
            f"Datos incompletos: se esperaban grupos de {tam_punto} valores "
            f"(índice + {num_vars} variables), pero hay {len(tokens)} tokens."
        )

    puntos = []
    for inicio in range(0, len(tokens), tam_punto):
        grupo = tokens[inicio:inicio + tam_punto]
        puntos.append([float(valor) for valor in grupo[1:]])

    if len(variables) != num_vars:
        raise ValueError(
            f"La cabecera declara {num_vars} variables, "
            f"pero se encontraron {len(variables)} nombres."
        )
    if not puntos:
        raise ValueError("El archivo no contiene puntos de simulación.")

    frecuencia_presente = 'frequency' in [nombre.lower() for nombre in variables]
    es_ruido_integrado = (
        'integrated noise' in plotname.lower()
        or (num_puntos == 1 and not frecuencia_presente)
    )
    if es_ruido_integrado:
        resultados = [
            (nombre, abs(puntos[0][indice]))
            for indice, nombre in enumerate(variables)
        ]
        return sorted(resultados, key=lambda x: x[1], reverse=True)

    frecuencias = [punto[0] for punto in puntos]
    resultados = []
    for indice_variable, nombre in enumerate(variables):
        if nombre.lower() == 'frequency':
            continue

        densidad = [punto[indice_variable] for punto in puntos]
        # Las variables *_spectrum son densidades (unidad/sqrt(Hz));
        # integrar PSD = densidad^2 produce el ruido RMS total.
        ruido_rms_cuadrado = sum(
            (frecuencias[i + 1] - frecuencias[i])
            * (densidad[i] ** 2 + densidad[i + 1] ** 2) / 2
            for i in range(len(frecuencias) - 1)
        )
        if ruido_rms_cuadrado < 0:
            raise ValueError(f"Frecuencias no crecientes para '{nombre}'.")
        resultados.append((nombre, math.sqrt(ruido_rms_cuadrado)))

    resultados_ordenados = sorted(resultados, key=lambda x: x[1], reverse=True)

    return resultados_ordenados

# --- RUTA DE TU ARCHIVO ---
ruta_raw = '/home/jaskun/simulations/ngspice/ckt_noise_2.raw'
ruta_csv_salida = '/home/jaskun/simulations/ngspice/ruido_ordenado.csv'

ranking = leer_y_ordenar_ruido_ngspice(ruta_raw)

# 1. Mostrar en pantalla
print(f"{'POS':<4} | {'COMPONENTE / FUENTE':<35} | {'RUIDO INTEGRADO'}")
print("-" * 62)
for i, (nombre, valor) in enumerate(ranking, start=1):
    print(f"{i:<4} | {nombre:<35} | {valor:.6e}")

# 2. Exportar a CSV (listo para Google Sheets / Excel)
with open(ruta_csv_salida, 'w', newline='', encoding='utf-8') as f_csv:
    writer = csv.writer(f_csv)
    writer.writerow(['Posicion', 'Variable', 'Ruido_RMS_Integrado'])
    for i, (nombre, valor) in enumerate(ranking, start=1):
        writer.writerow([i, nombre, valor])

print(f"\nArchivo CSV guardado en: {ruta_csv_salida}")


POS  | COMPONENTE / FUENTE                 | RUIDO INTEGRADO
--------------------------------------------------------------
1    | @n.x1.xmg.nsg13_lv_pmos[gm]         | 3.285577e-02
2    | @n.x1.xmg.nsg13_lv_pmos[gds]        | 5.031640e-03
3    | i(@n.x1.xmg.nsg13_lv_pmos[ids])     | 5.026947e-03
4    | v(inoise_total)                     | 3.158755e-03
5    | v(inoise_total_n.x1.xm1.nsg13_lv_nmos) | 1.355392e-03
6    | v(inoise_total_n.x1.xm2.nsg13_lv_nmos) | 1.351106e-03
7    | v(inoise_total_n.x1.xm1.nsg13_lv_nmos_flicker) | 1.325088e-03
8    | v(inoise_total_n.x1.xm4.nsg13_lv_pmos) | 1.321427e-03
9    | v(inoise_total_n.x1.xm2.nsg13_lv_nmos_flicker) | 1.320578e-03
10   | v(inoise_total_n.x1.xm4.nsg13_lv_pmos_flicker) | 1.300365e-03
11   | v(inoise_total_n.x1.xm5.nsg13_lv_nmos) | 1.234273e-03
12   | v(inoise_total_n.x1.xm5.nsg13_lv_nmos_flicker) | 1.234230e-03
13   | v(inoise_total_n.x1.xm3.nsg13_lv_pmos) | 1.226841e-03
14   | v(inoise_total_n.x1.xm3.nsg13_lv_pmos_flicker) | 1.20734